# Fabric API tests (fabric-cicd `0.2.0`)

This notebook targets the latest docs for `https://microsoft.github.io/fabric-cicd/0.2.0/`.
Destructive operations are disabled by default.


In [1]:
# Cell 1 - Install required packages in the notebook kernel.
# Use `%pip` in notebooks so packages are installed in the active kernel environment.
%pip install -q fabric-cicd==0.2.0 azure-identity


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 - Import dependencies and print API metadata so we verify runtime assumptions.
import os
from pathlib import Path

from azure.identity import AzureCliCredential, ClientSecretCredential, DefaultAzureCredential

from fabric_cicd import (
    FabricWorkspace,
    append_feature_flag,
    publish_all_items,
    unpublish_all_orphan_items,
)
from fabric_cicd._common._exceptions import InputError
from fabric_cicd.constants import FeatureFlag
from fabric_cicd import constants

print('DEFAULT_API_ROOT_URL:', constants.DEFAULT_API_ROOT_URL)
print('Accepted item type count:', len(constants.ACCEPTED_ITEM_TYPES))
print('OrgApp supported:', 'OrgApp' in constants.ACCEPTED_ITEM_TYPES)


DEFAULT_API_ROOT_URL: https://api.powerbi.com
Accepted item type count: 25
OrgApp supported: False


In [3]:
# Cell 3 - Load test configuration from environment variables and choose auth mode.
# Set these before running:
# - FABRIC_WORKSPACE_ID or FABRIC_WORKSPACE_NAME (target)
# - FABRIC_SOURCE_WORKSPACE_NAME (optional, for read-only source/target comparison)
# - FABRIC_REPOSITORY_DIRECTORY
# - FABRIC_ENVIRONMENT
# - FABRIC_AUTH_MODE = azure_cli | service_principal | default
# Optional but recommended for azure_cli mode:
# - AZURE_TENANT_ID
# - AZURE_CONFIG_DIR (example: .\.az-fabric)
WORKSPACE_ID = os.getenv('FABRIC_WORKSPACE_ID', '').strip()
WORKSPACE_NAME = os.getenv('FABRIC_WORKSPACE_NAME', 'konsolidator').strip()
SOURCE_WORKSPACE_NAME = os.getenv('FABRIC_SOURCE_WORKSPACE_NAME', 'konsolidator_dev').strip()
REPOSITORY_DIRECTORY = os.getenv('FABRIC_REPOSITORY_DIRECTORY', str(Path.cwd()))
ENVIRONMENT = os.getenv('FABRIC_ENVIRONMENT', 'MIGRATION')
AUTH_MODE = os.getenv('FABRIC_AUTH_MODE', 'azure_cli').strip().lower()
AZURE_TENANT_ID = os.getenv('AZURE_TENANT_ID', '1dc7fe1c-3fd1-428f-b745-de881b406952').strip()

# Ensure an isolated Azure CLI profile can be used from VS Code/Jupyter kernels.
if not os.getenv('AZURE_CONFIG_DIR'):
    local_az_cfg = Path.cwd() / '.az-fabric'
    if local_az_cfg.exists():
        os.environ['AZURE_CONFIG_DIR'] = str(local_az_cfg)

if AUTH_MODE == 'service_principal':
    credential = ClientSecretCredential(
        tenant_id=os.environ['AZURE_TENANT_ID'],
        client_id=os.environ['AZURE_CLIENT_ID'],
        client_secret=os.environ['AZURE_CLIENT_SECRET'],
    )
elif AUTH_MODE == 'default':
    credential = DefaultAzureCredential()
else:
    # Pass tenant explicitly to avoid account ambiguity in Azure CLI.
    credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID)

print('AUTH_MODE:', AUTH_MODE)
print('AZURE_TENANT_ID:', AZURE_TENANT_ID)
print('AZURE_CONFIG_DIR:', os.getenv('AZURE_CONFIG_DIR', '<not set>'))
print('TARGET_WORKSPACE_NAME:', WORKSPACE_NAME)
print('SOURCE_WORKSPACE_NAME:', SOURCE_WORKSPACE_NAME)
print('REPOSITORY_DIRECTORY:', REPOSITORY_DIRECTORY)
print('ENVIRONMENT:', ENVIRONMENT)
print('Using identifier:', 'workspace_id' if WORKSPACE_ID else 'workspace_name')


AUTH_MODE: azure_cli
REPOSITORY_DIRECTORY: c:\Users\marti\OneDrive - BioNordic\Desktop\Fabric API
ENVIRONMENT: dev
Using identifier: workspace_name


In [4]:
# Cell 4 - Validate authentication by requesting a Fabric access token.
# If this fails in VS Code, restart kernel after setting env vars in terminal and re-run from top.
scope = 'https://api.fabric.microsoft.com/.default'
try:
    token = credential.get_token(scope)
except Exception as ex:
    print('Token acquisition failed:', type(ex).__name__)
    print(ex)
    raise

assert token and token.token, 'Failed to get access token'
print('Token acquired. Expires on:', token.expires_on)


Token acquired. Expires on: 1771594660


In [5]:
# Cell 5 - Negative test: ensure invalid workspace IDs are rejected by FabricWorkspace validation.
try:
    FabricWorkspace(
        workspace_id='not-a-guid',
        repository_directory=REPOSITORY_DIRECTORY,
        environment=ENVIRONMENT,
        token_credential=credential,
    )
    raise AssertionError('Expected InputError for invalid workspace_id')
except InputError as ex:
    print('PASS invalid workspace_id validation:', ex)


[info]   13:34:14 - Executing as User 'martin.falces@serascandia.com'


PASS invalid workspace_id validation: The provided workspace_id is not a valid guid.


In [ ]:
# Cell 6 - Create a real FabricWorkspace object for your target workspace.
# item_type_in_scope can be adjusted to only the item types you deploy.
fw_kwargs = dict(
    repository_directory=REPOSITORY_DIRECTORY,
    environment=ENVIRONMENT,
    item_type_in_scope=['Notebook', 'DataPipeline', 'Environment', 'Lakehouse'],
    token_credential=credential,
)
if WORKSPACE_ID:
    fw_kwargs['workspace_id'] = WORKSPACE_ID
else:
    fw_kwargs['workspace_name'] = WORKSPACE_NAME

workspace = FabricWorkspace(**fw_kwargs)
print('Resolved workspace_id:', workspace.workspace_id)
print('Items in scope:', workspace.item_type_in_scope)
print('Parameter file path:', workspace.parameter_file_path)


In [ ]:
# Cell 6b - Optional read-only migration check: compare source vs target inventory by item type.
# This does not deploy anything; it only reads metadata from both workspaces.
from collections import Counter

if SOURCE_WORKSPACE_NAME:
    source_workspace = FabricWorkspace(
        workspace_name=SOURCE_WORKSPACE_NAME,
        repository_directory=REPOSITORY_DIRECTORY,
        environment=ENVIRONMENT,
        token_credential=credential,
    )

    src_items = source_workspace.endpoint.invoke(
        method='GET',
        url=f"{constants.DEFAULT_API_ROOT_URL}/v1/workspaces/{source_workspace.workspace_id}/items",
    )['body'].get('value', [])

    tgt_items = workspace.endpoint.invoke(
        method='GET',
        url=f"{constants.DEFAULT_API_ROOT_URL}/v1/workspaces/{workspace.workspace_id}/items",
    )['body'].get('value', [])

    src_counts = Counter(item.get('type', 'Unknown') for item in src_items)
    tgt_counts = Counter(item.get('type', 'Unknown') for item in tgt_items)

    print('Source workspace:', SOURCE_WORKSPACE_NAME, source_workspace.workspace_id)
    print('Target workspace:', WORKSPACE_NAME or WORKSPACE_ID, workspace.workspace_id)
    print('Source item count:', len(src_items))
    print('Target item count:', len(tgt_items))
    print('Source by type:', dict(src_counts))
    print('Target by type:', dict(tgt_counts))
else:
    print('SOURCE_WORKSPACE_NAME not set; skipping source/target comparison.')


In [ ]:
# Cell 7 - Read-only smoke tests against Fabric REST endpoints used by fabric-cicd.
workspace_details = workspace.endpoint.invoke(
    method='GET',
    url=f"{constants.DEFAULT_API_ROOT_URL}/v1/workspaces/{workspace.workspace_id}",
)
items_list = workspace.endpoint.invoke(
    method='GET',
    url=f"{constants.DEFAULT_API_ROOT_URL}/v1/workspaces/{workspace.workspace_id}/items",
)

assert workspace_details['status_code'] == 200, workspace_details
assert items_list['status_code'] == 200, items_list

print('Workspace name:', workspace_details['body'].get('displayName'))
print('Item count:', len(items_list['body'].get('value', [])))


In [ ]:
# Cell 8 - Safe feature-flag enforcement test for selective deployment.
# In 0.2.0, this still requires ENABLE_EXPERIMENTAL_FEATURES + ENABLE_ITEMS_TO_INCLUDE.
try:
    publish_all_items(workspace, items_to_include=['Example.Notebook'])
    raise AssertionError('Expected InputError because feature flags are missing')
except InputError as ex:
    print('PASS feature flag check:', ex)


In [ ]:
# Cell 9 - Optional publish smoke test (disabled by default).
# Enable with care: this can create/update items in your target workspace.
RUN_PUBLISH = False
if RUN_PUBLISH:
    append_feature_flag(FeatureFlag.ENABLE_RESPONSE_COLLECTION.value)
    responses = publish_all_items(workspace)
    print('Publish responses collected:', bool(responses))
    if responses:
        print('Response item types:', list(responses.keys()))
else:
    print('Skipped publish test. Set RUN_PUBLISH=True to execute.')


In [ ]:
# Cell 10 - Optional unpublish smoke test (disabled by default).
# Enable with care: this can delete orphaned deployed items from the workspace.
RUN_UNPUBLISH = False
if RUN_UNPUBLISH:
    unpublish_all_orphan_items(workspace, item_name_exclude_regex='^$')
    print('Unpublish operation completed')
else:
    print('Skipped unpublish test. Set RUN_UNPUBLISH=True to execute.')
